In [ ]:
import os
import time
import math
import json
import random
import pandas as pd
import numpy as np
import scipy as sp
import pylab as plt
import seaborn as sns
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
from sklearn.linear_model import LinearRegression

In [ ]:
%load_ext autoreload
%autoreload 2
import src.count_utils as utils

In [ ]:
import jupyter_black

jupyter_black.load()

In [ ]:
plt.style.use("../src/mpl_style.txt")

### Excess word list with frequency cutoff (non-lemmatized)
To determine the optimal rare word list that yields the best LLM use estimate,
we try different frequency cutoffs. The base list consists of all excess style 
words from the 2024 pubmed data. For each cutoff value, only words with frequency 
lower than the cutoff (average 2024) are considered into the list 
of rare words. 

In this version, word lemmata are being ignored and each 
version of a word is treated as a separate entity, e.g "delves" != "delve"
This could be a meaningful difference, because LLMs might have preferences
for certain versions of a word, but not others (how does tokenization work 
for ChatGPT? is the base token for "delve", "delves", "delving" the same?
is this relevant? despite having the same base tokens, different variants
occurr in different semantic and syntactic contexts, so to differentiate
between variants could still be informative, despite being based on the 
same token)

Procedure:
- get all 2024 excess words from the previous paper 
- compute the non-lemmatized average 2024 frequencies for the excess words 
(if they are present in the section vocabulary)
- filter list to only include words with average 2024/2025 frequency below cutoff
- group counts for all words in list
- use new counts to compute frequency, diffs, ratios and projection 
- repeat for every cutoff and section

Open questions:
- should there be a lower bound for frequencies (i.e. 1e-4) as there was in the 
paper? The excess words were determined with a lower bound, so they already 
exclude artefacts

In [ ]:
VERSION = "research-article_aimrd_f"
BASELINE = "baseline_2026-01-23"
RESULTS_PATH = os.path.join("../data/results/", BASELINE, VERSION)
secs = next(os.walk(RESULTS_PATH))[1]
# values are either "" or a section name, if not "", only use frequs of the given
# section to compute excess word lists to be used for all sections
vocab_version = ""

In [ ]:
years = np.arange(2000, 2026)
months = np.arange(1, 13)
with open(os.path.join(RESULTS_PATH, "filters.json")) as f:
    filters = json.load(f)
dates = pd.to_datetime(filters["all_dates"])

In [ ]:
excess_words = utils.excess_style_words_2024
len(excess_words)

In [ ]:
cutoffs = [
    0.0002,
    0.0005,
    0.001,
    0.002,
    0.005,
    0.01,
    0.02,
    0.03,
    0.04,
    0.05,
    0.07,
    0.1,
    0.15,
    0.2,
    0.3,
    0.4,
    0.5,
    0.7,
    1.0,
]
fig, ax = plt.subplots(figsize=(4, 1), layout="constrained")
plt.scatter(cutoffs, [1] * len(cutoffs))
ax.set_xscale("log")

In [ ]:
excess_words_cutoff = {}
cutoff_counts = np.zeros((len(secs), len(cutoffs)), dtype=int)

for i, sec in enumerate(secs):
    if not vocab_version == "":
        if not sec == vocab_version:
            continue

    excess_words_cutoff[sec] = {}

    X = sp.sparse.load_npz(os.path.join(RESULTS_PATH, sec, f"count_{sec}.pkl.npz"))

    words = np.load(
        os.path.join(RESULTS_PATH, sec, f"words_{sec}.pkl.npy"), allow_pickle=True
    )

    ind_words = np.isin(words, excess_words)
    ind = dates.year == 2024

    counts = np.array(np.sum(X[ind, :][:, ind_words], axis=0)).ravel()
    totals = np.sum(ind)

    freqs_2024 = (counts + 1) / (totals + 1)

    for j, cutoff in enumerate(cutoffs):

        ind_excess = freqs_2024 < cutoff

        cutoff_counts[i, j] = np.sum(ind_excess)
        excess_words_cutoff[sec][cutoff] = words[ind_words][ind_excess]

    # collect number of papers per year
    if sec == "abstract":
        totals_yearly = {}
        for year in range(2018, 2026):
            totals_yearly[year] = np.sum(dates.year == year)


pd.DataFrame(cutoff_counts, columns=cutoffs, index=secs)

In [ ]:
excess_words_cutoff.keys()

In [ ]:
totals_yearly

In [ ]:
freqs_dfs = {}

for sec in secs:
    if vocab_version == "":
        freqs_dfs[sec] = utils.get_cutoff_frequency(
            RESULTS_PATH,
            sec,
            excess_words_cutoff[sec],
            lemmatized="_non-lemmatized",
        )
    else:
        freqs_dfs[sec] = utils.get_cutoff_frequency(
            RESULTS_PATH,
            sec,
            excess_words_cutoff[vocab_version],
            lemmatized=f"_non-lemmatized_{vocab_version}_only",
        )

In [ ]:
totals = [totals_yearly[y] for y in freqs_dfs["abstract"]["time"]]

In [ ]:
p = 0.5
q = 0.7
n = 120000
var_p = p * (1 - p) / n
math.sqrt(utils.var_g(p, var_p, q, n)[0][0])

In [ ]:
utils.se(p, q, n, "binomial")

In [ ]:
for sec in secs:
    p = freqs_dfs[sec]["projection"]
    q = freqs_dfs[sec]["frequency"]
    reg_se = freqs_dfs[sec]["regression se"]
    freqs_dfs[sec]["binomial se"] = list(map(lambda p, n: p * (1 - p) / n, p, totals))
    freqs_dfs[sec]["binomial se (ds)"] = list(
        map(utils.se, p, q, totals, ["binomial"] * len(p))
    )
    freqs_dfs[sec]["regression se (ds)"] = list(
        map(utils.se, p, q, totals, ["regression"] * len(p), reg_se)
    )
    freqs_dfs[sec]["combined se (ds)"] = list(
        map(utils.se, p, q, totals, ["both"] * len(p), reg_se)
    )
    freqs_dfs[sec]["regression se (delta)"] = list(
        map(utils.se, p, q, totals, ["regression"] * len(p), reg_se, ["delta"] * len(p))
    )

In [ ]:
# filter out cutoffs with too high base frequency
freqs_dfs_filtered = {}
for sec in secs:
    remove = freqs_dfs[sec][freqs_dfs[sec]["projection"] >= 0.999]["cutoff"].unique()
    freqs_dfs_filtered[sec] = freqs_dfs[sec][
        freqs_dfs[sec]["cutoff"].apply(lambda x: x not in remove)
    ].copy()

In [ ]:
freqs_dfs["introduction"]

In [ ]:
freqs_dfs_filtered["introduction"]

In [ ]:
fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(5, 5), layout="constrained")

hue_norm = LogNorm()
palette = "flare"
data = freqs_dfs["results"].copy()
data["time"] = list(map(int, data["time"]))
for i, var in enumerate(["frequency", "diff", "usage estimate"]):
    ax = axs.flatten()[i]

    ax.axvline(x=2022 + (11 / 12), linestyle="--", color="black", alpha=0.3)
    sns.lineplot(
        data=data,
        x="time",
        y=var,
        hue="cutoff",
        hue_norm=hue_norm,
        palette=palette,
        ax=ax,
        legend=False,
    )

    if var == "frequency":
        sns.lineplot(
            data=data,
            x="time",
            y="projection",
            hue="cutoff",
            hue_norm=LogNorm(),
            palette=palette,
            ax=ax,
            alpha=0.5,
            linestyle="-.",
            legend=False,
        )

    if var == "usage estimate":
        ax.set_ylim(-0.01, 1.01)

    if var == "diff":
        sm = plt.cm.ScalarMappable(cmap=palette, norm=hue_norm)
        cax = fig.add_axes(
            [
                ax.get_position().x1 + 0.1,
                ax.get_position().y0,
                0.02,
                ax.get_position().height * 2,
            ]
        )
        ax.figure.colorbar(sm, cax=cax)

In [ ]:
"""freqs_df_agg = {}
for sec in secs:
    df = freqs_dfs[sec].copy()
    df["time"] = list(map(math.floor, df["time"]))
    df = df.groupby(["time", "cutoff"]).mean().reset_index()
    freqs_df_agg[sec] = df"""

In [ ]:
"""freqs_df_filtered_agg = {}
for sec in secs:
    df = freqs_dfs_filtered[sec].copy()
    df["time"] = list(map(math.floor, df["time"]))
    df = df.groupby(["time", "cutoff"]).mean().reset_index()
    freqs_df_filtered_agg[sec] = df"""

In [ ]:
freqs_dfs["abstract"][freqs_dfs["abstract"]["time"] == 2025]

In [ ]:
freqs_dfs["full"][freqs_dfs["full"]["time"] == "2025"].columns

In [ ]:
# ensure correct order
sections = ["abstract", "introduction", "methods", "results", "discussion", "full"]
target_year = 2023
se_thr_low = 0.01
se_thr_high = 0.025

fig = plt.figure(figsize=(7, 6), layout="constrained")
subfigs = fig.subfigures(3, 2)  # outer grid

for subfig, sec in zip(subfigs.flat, sections):
    f = freqs_dfs[sec][freqs_dfs[sec]["time"] == target_year]
    f_filt = freqs_dfs_filtered[sec][freqs_dfs_filtered[sec]["time"] == target_year]
    if len(np.where(f_filt["regression se (ds)"] > se_thr_high)[0]) > 0:
        high_error = np.where(f_filt["regression se (ds)"] > se_thr_high)[0][0]
    else:
        high_error = len(f_filt)

    axs = subfig.subplots(1, 2)  # inner grid

    subfig.suptitle(sec)

    # frequency plot
    axs[0].plot(
        f["cutoff"],
        f["frequency"],
        ".-",
        clip_on=False,
        label="Observed\nfrequency\n($q$)",
        markersize=2.5,
    )
    axs[0].errorbar(
        f["cutoff"],
        f["projection"],
        yerr=np.where(
            f["regression se"] > se_thr_low, f["regression se"], np.nan
        ),  # ±1 standard errors
        fmt=".-",  # np.where(yerr >= threshold, yerr, np.nan)
        # ecolor="grey",
        capsize=2,
        label="Expected\nfrequency\n($\\hat{p}_{human}$)",
        clip_on=False,
        markersize=2.5,
    )
    axs[0].set_xscale("log")
    axs[0].set_xlim([1e-4, 1])
    axs[0].set_ylim([0, 1])
    axs[0].set_ylabel("Frequency")
    if sec == "abstract":
        axs[0].legend()
    if sec in ["discussion", "full"]:
        axs[0].set_xlabel("Threshold")

    # usage estimate plot

    axs[1].errorbar(
        f_filt["cutoff"][high_error - 1 :],
        f_filt["usage estimate"][high_error - 1 :],
        yerr=f_filt["regression se (ds)"][high_error - 1 :],  # ±1 standard errors
        fmt=".-",
        color="silver",
        capsize=2,
        markersize=2.5,
        zorder=0.5,
    )
    axs[1].errorbar(
        f_filt["cutoff"][:high_error],
        f_filt["usage estimate"][:high_error],
        yerr=np.where(
            f_filt["regression se (ds)"][:high_error] > se_thr_low,
            f_filt["regression se (ds)"][:high_error],
            np.nan,
        ),  # ±1 standard errors
        fmt="r.-",
        capsize=2,
        label="LLM-use estimate ($\\beta_{lb}$)",
        markersize=2.5,
    )
    axs[1].errorbar(
        f_filt["cutoff"],
        f_filt["diff"],
        yerr=np.where(
            f_filt["regression se (delta)"] > se_thr_low,
            f_filt["regression se (delta)"],
            np.nan,
        ),  # ±1 standard errors
        fmt="k.-",
        capsize=2,
        label="Frequency gap ($\\Delta$)",
        clip_on=False,
        markersize=2.5,
    )

    axs[1].set_xscale("log")
    axs[1].set_xlim([1e-4, 1.15])
    axs[1].set_ylim([0, 1])
    axs[1].set_ylabel("LLM-use ($\\beta$)")
    if sec == "abstract":
        axs[1].legend()
    if sec in ["discussion", "full"]:
        axs[1].set_xlabel("Threshold")

plt.suptitle(f"{target_year} estimated LLM usage")
plt.savefig(
    os.path.join(
        "../results/plots",
        BASELINE,
        VERSION,
        f"5y_{target_year}_lower_bound_non_lemmatized_{vocab_version}.png",
    ),
    dpi=300,
)

In [ ]:
len(np.where(f_filt["combined se (ds)"] > se_thr_high)[0])

In [ ]:
# ensure correct order
sections = ["abstract", "introduction", "methods", "results", "discussion", "full"]

fig, axs = plt.subplots(3, 2, figsize=(7, 6), layout="constrained")

for ax, sec in zip(axs.flat, sections):
    f = freqs_dfs[sec][freqs_dfs[sec]["time"] == 2025]
    f_filt = freqs_dfs_filtered[sec][freqs_dfs_filtered[sec]["time"] == 2025]

    # ax.title(sec)

    # frequency plot
    ax.plot(
        f["cutoff"], f["frequency"], ".-", clip_on=True, label="Observed\nfrequency"
    )
    # axs[0].plot(
    #    f["cutoff"], f["projection"], ".-", clip_on=False, label="Expected\nfrequency"
    # )
    ax.errorbar(
        f["cutoff"],
        f["projection"],
        yerr=list(map(lambda x: x * 2, f["regression se"])),  # ±2 standard errors
        fmt=".-",
        ecolor="grey",
        capsize=2,
        label="Expected\nfrequency",
    )
    ax.axhline(0.99, color="grey", linestyle="--")
    ax.set_xscale("log")
    ax.set_xlim([0.15, 1.01])
    ax.set_ylim([0.85, 1.01])
    ax.set_ylabel("Frequency")
    ax.set_xlabel("Threshold")
    ax.set_title(sec)
    if sec == "abstract":
        ax.legend()

plt.savefig(
    os.path.join(
        "../results/plots",
        BASELINE,
        VERSION,
        f"close-up_5y_{target_year}_lower_bound_non_lemmatized_{vocab_version}.png",
    ),
    dpi=300,
)

In [ ]:
# ensure correct order
sections = ["abstract", "introduction", "methods", "results", "discussion", "full"]
target_year = 2025

fig = plt.figure(figsize=(8, 10), layout="constrained")
subfigs = fig.subfigures(6, 1)  # outer grid

colors = ["darkturquoise", "mediumvioletred", "mediumslateblue"]

for subfig, sec in zip(subfigs, sections):

    f = freqs_dfs[sec][freqs_dfs[sec]["time"] == target_year]
    f_filt = freqs_dfs_filtered[sec][freqs_dfs_filtered[sec]["time"] == target_year]

    axs = subfig.subplots(1, 5)  # inner grid

    subfig.suptitle(sec)

    # frequency errors
    for i, s in enumerate(["regression", "binomial"]):
        axs[0].plot(
            f_filt["cutoff"],
            f_filt[f"{s} se"],
            ".-",
            clip_on=False,
            label=s,
            color=colors[i],
        )

    # axs[0].set_xlim([1e-4, 1])
    # axs[0].set_ylim([0, 1])
    axs[0].set_ylabel("Frequency SE")

    # usage estimate errors
    for i, s in enumerate(["regression", "binomial", "combined"]):
        axs[1].plot(
            f_filt["cutoff"],
            f_filt[f"{s} se (ds)"],
            ".-",
            clip_on=False,
            label=s,
            color=colors[i],
        )

    # axs[1].set_xlim([1e-4, 1])
    # axs[1].set_ylim([0, 1])
    axs[1].set_ylabel("Usage Estimate SE")

    # usage estimate plot
    for i, s in enumerate(["regression", "binomial", "combined"]):
        axs[i + 2].errorbar(
            f["cutoff"],
            f["usage estimate"],
            yerr=f[f"{s} se (ds)"],  # ±1 standard errors
            fmt=".-",
            color="grey",
            ecolor=colors[i],
            capsize=2,
            label=f"usage estimate with\n{s} error bars",
            # clip_on=False,
        )
        axs[i + 2].set_xlim([1e-4, 1.15])
        axs[i + 2].set_ylim([0, 1])
        axs[i + 2].set_ylabel("usage estimate")

    for i in range(5):
        axs[i].set_xscale("log")
        if sec == "abstract":
            axs[i].legend()
        if sec == "full":
            axs[i].set_xlabel("Threshold")


plt.suptitle(f"{target_year} estimated LLM usage")

plt.savefig(
    os.path.join(
        "../results/plots",
        BASELINE,
        VERSION,
        f"error_comp_5y_{target_year}_lower_bound_non_lemmatized_{vocab_version}.png",
    ),
    dpi=300,
)

find the cutoff yielding the maximum usage estimate (given that p<0.999 and the error<0.025) for each year and section.
Use the cutoff that produces the maximum for 2025 as the default list to compute group frequencies.

In [ ]:
max_est = {}
for sec in sections:
    max_est[sec] = {}
    for year in [2023, 2024, 2025]:
        f_filt = freqs_dfs_filtered[sec][freqs_dfs_filtered[sec]["time"] == year]
        if len(np.where(f_filt["regression se (ds)"] > se_thr_high)[0]) > 0:
            high_error = np.where(f_filt["regression se (ds)"] > se_thr_high)[0][0]
        else:
            high_error = len(f_filt)

        i = int(np.argmax(f_filt["usage estimate"][:high_error]))
        max_est[sec][f"usage ({year})"] = list(f_filt["usage estimate"])[i]
        max_est[sec][f"sdev ({year})"] = list(f_filt["regression se (ds)"])[i]
        max_est[sec][f"cutoff ({year})"] = list(f_filt["cutoff"])[i]

In [ ]:
df = pd.DataFrame(max_est).T
df

In [ ]:
f_filt = freqs_dfs_filtered[sec][freqs_dfs_filtered[sec]["time"] == 2025]
f_filt[f_filt["cutoff"] == 0.15]["usage estimate"]

In [ ]:
print(df.to_latex(float_format="%.2f"))

In [ ]:
f_filt[f_filt["cutoff"] == 0.1]["usage estimate"]

In [ ]:
df.loc["abstract"]["cutoff (2025)"]

In [ ]:
max_est_fixed = {}
for sec in sections:
    max_est_fixed[sec] = {}

    for year in [2023, 2024, 2025]:
        f_filt = freqs_dfs_filtered[sec][freqs_dfs_filtered[sec]["time"] == year]
        c = df.loc[sec]["cutoff (2025)"]
        f_filt = f_filt[f_filt["cutoff"] == c]

        max_est_fixed[sec][f"usage ({year})"] = np.float64(f_filt["usage estimate"])
        max_est_fixed[sec][f"sdev ({year})"] = np.float64(f_filt["regression se (ds)"])

In [ ]:
df2 = pd.DataFrame(max_est_fixed).T
df2["cutoff"] = df["cutoff (2025)"]
df2

In [ ]:
print(df2.to_latex(float_format="%.2f"))

In [ ]:
for sec in secs:
    co1 = max_est[sec]["cutoff (2025)"]
    co1_ind = np.where(np.asarray(cutoffs) == co1)[0][0]
    co2 = cutoffs[co1_ind - 1]
    co3 = cutoffs[co1_ind - 2]
    ew1 = excess_words_cutoff[sec][co1]
    ew2 = excess_words_cutoff[sec][co2]
    ew3 = excess_words_cutoff[sec][co3]
    np.save(
        os.path.join(RESULTS_PATH, sec, f"optimized_excess_words_{sec}.pkl"),
        ew1,
        allow_pickle=True,
    )
    np.save(
        os.path.join(RESULTS_PATH, sec, f"suboptimized_excess_words_{sec}.pkl"),
        ew2,
        allow_pickle=True,
    )
    np.save(
        os.path.join(RESULTS_PATH, sec, f"subsuboptimized_excess_words_{sec}.pkl"),
        ew3,
        allow_pickle=True,
    )
    print(f"{sec}: {len(ew1)}({len(ew2)}) excess words at optimal cutoff {co1}({co2})")

In [ ]:
co

In [ ]:
i = np.where(np.asarray(cutoffs) == co)[0][0]
i

In [ ]:
cutoffs[i - 1]

In [ ]:
excess_words_cutoff["abstract"][max_est["abstract"]["cutoff (2025)"]]

In [ ]:
# get excess words sorted by abstract freqs
sec = "abstract"
X = sp.sparse.load_npz(os.path.join(RESULTS_PATH, sec, f"count_{sec}.pkl.npz"))

words = np.load(
    os.path.join(RESULTS_PATH, sec, f"words_{sec}.pkl.npy"), allow_pickle=True
)

ind_words = np.isin(words, excess_words)
ind = dates.year == 2024

counts = np.array(np.sum(X[ind, :][:, ind_words], axis=0)).ravel()
totals = np.sum(ind)

freqs_2024 = (counts + 1) / (totals + 1)

In [ ]:
ew_df = pd.DataFrame(zip(freqs_2024, words[ind_words]), columns=["freq", "ew"])
ew_df = ew_df.sort_values("freq")
print(list(ew_df["ew"]))

In [ ]:
ew_df[ew_df["freq"] >= 0.1]